# 3D Helmholtz PFDM — exact-boundary linear-algebra scaling

This notebook uses **exactly the same full-grid finite-difference observation matrix and noise model as `main(1).ipynb`**:

\[
A = H K H^\top + \Sigma + \varepsilon I,
\]

where boundary rows of \(H\) are identity observations with zero noise, and interior rows use the 7-point Helmholtz stencil with noise variance \(C h^{2\tau}\).

The hyperparameters are fixed to the values selected by the proposed method at \(N_x=5\):

\[
\nu=1.5,\qquad
\ell=0.333333,\qquad
\sigma_f=1.02239,\qquad
C=327.332,\qquad
\tau=2,\qquad
k=1.
\]

We compare the **same linear system** using:

1. dense Cholesky,
2. dense CG,
3. matrix-free CG with sparse \(H\) + BTTTB/3D FFT covariance matvecs.


In [1]:
using LinearAlgebra
using SparseArrays
using FFTW
using SpecialFunctions
using IterativeSolvers
using LinearMaps
using Plots
using Statistics
using Printf


In [2]:
# 3次元 Matérnカーネル（入力 x1, x2 ∈ R³）
function matern_kernel_3d(x1, x2; ν=1.5, ℓ=1.0, σf=1.0)
    r = sqrt(sum((x1 .- x2).^2))
    r == 0 && return σf^2
    z = sqrt(2ν) * r / ℓ
    if ν == 0.5
        return σf^2 * exp(-z)
    elseif ν == 1.5
        return σf^2 * (1 + z) * exp(-z)
    elseif ν == 2.5
        return σf^2 * (1 + z + z^2/3) * exp(-z)
    else
        return σf^2 * (2^(1-ν)/gamma(ν)) * z^ν * besselk(ν, z)
    end
end

# 3D observation matrix on the FULL grid, including boundaries.
#
# Boundary rows: u = 0 exactly  -> identity row
# Interior rows: (-Delta_h + k^2 I)u = f
#
# This is the direct 3D analogue of the 1D construction
#   Sigma_diag = [0, interior noise..., 0].
function build_H_3d_helmholtz_full(Nx, h, k)
    # Nx = number of INTERIOR points per coordinate direction.
    nside = Nx + 2
    Nall  = nside^3

    index(i,j,l) = i + (j-1)*nside + (l-1)*nside^2

    rows = Int[]
    cols = Int[]
    vals = Float64[]
    is_boundary = falses(Nall)

    for l in 1:nside, j in 1:nside, i in 1:nside
        q = index(i,j,l)

        bdry = (i == 1 || i == nside ||
                j == 1 || j == nside ||
                l == 1 || l == nside)

        is_boundary[q] = bdry

        if bdry
            # Exact homogeneous Dirichlet condition.
            push!(rows,q); push!(cols,q); push!(vals,1.0)
        else
            # Helmholtz PDE row on an interior point.
            push!(rows,q); push!(cols,q); push!(vals,6.0/h^2 + k^2)
            push!(rows,q); push!(cols,index(i-1,j,l)); push!(vals,-1.0/h^2)
            push!(rows,q); push!(cols,index(i+1,j,l)); push!(vals,-1.0/h^2)
            push!(rows,q); push!(cols,index(i,j-1,l)); push!(vals,-1.0/h^2)
            push!(rows,q); push!(cols,index(i,j+1,l)); push!(vals,-1.0/h^2)
            push!(rows,q); push!(cols,index(i,j,l-1)); push!(vals,-1.0/h^2)
            push!(rows,q); push!(cols,index(i,j,l+1)); push!(vals,-1.0/h^2)
        end
    end

    H = sparse(rows,cols,vals,Nall,Nall)
    return H, is_boundary
end

# Observation vector on the full grid:
# boundary -> 0, interior -> f.
function make_y_3d_full(xfull, k, is_boundary)
    y = zeros(length(xfull))
    for q in eachindex(xfull)
        if !is_boundary[q]
            x,yq,z = xfull[q]
            y[q] = f_func(x,yq,z,k)
        end
    end
    return y
end

# BTTTB セットアップ: 3D循環埋め込み + 3D FFT
function btttb_fft_setup(k_col_arr, Nx, Ny, Nz)
    # k_col_arr[i,j,l] = k(x1, x_{i,j,l}) for i in 1:Nx, j in 1:Ny, l in 1:Nz
    C = zeros(2Nx, 2Ny, 2Nz)
    C[1:Nx, 1:Ny, 1:Nz] .= k_col_arr
    # x方向の折り返し
    Nx > 1 && (C[Nx+2:2Nx, 1:Ny, 1:Nz] .= k_col_arr[Nx:-1:2, :, :])
    # y方向の折り返し
    Ny > 1 && (C[1:Nx, Ny+2:2Ny, 1:Nz] .= k_col_arr[:, Ny:-1:2, :])
    # z方向の折り返し
    Nz > 1 && (C[1:Nx, 1:Ny, Nz+2:2Nz] .= k_col_arr[:, :, Nz:-1:2])
    # xy折り返し
    Nx > 1 && Ny > 1 && (C[Nx+2:2Nx, Ny+2:2Ny, 1:Nz] .= k_col_arr[Nx:-1:2, Ny:-1:2, :])
    # xz折り返し
    Nx > 1 && Nz > 1 && (C[Nx+2:2Nx, 1:Ny, Nz+2:2Nz] .= k_col_arr[Nx:-1:2, :, Nz:-1:2])
    # yz折り返し
    Ny > 1 && Nz > 1 && (C[1:Nx, Ny+2:2Ny, Nz+2:2Nz] .= k_col_arr[:, Ny:-1:2, Nz:-1:2])
    # xyz折り返し
    Nx > 1 && Ny > 1 && Nz > 1 && (C[Nx+2:2Nx, Ny+2:2Ny, Nz+2:2Nz] .= k_col_arr[Nx:-1:2, Ny:-1:2, Nz:-1:2])
    return fft(fft(fft(C, 1), 2), 3)
end

function btttb_matvec(k_fft3, v, Nx, Ny, Nz)
    V     = reshape(v, Nx, Ny, Nz)
    V_pad = zeros(2Nx, 2Ny, 2Nz)
    V_pad[1:Nx, 1:Ny, 1:Nz] .= V
    result = real.(ifft(ifft(ifft(
        fft(fft(fft(V_pad, 1), 2), 3) .* k_fft3,
    1), 2), 3))
    return vec(result[1:Nx, 1:Ny, 1:Nz])
end

# 真の解・右辺: -Δu + k^2*u = f
# u(x,y,z) = sin(πx)sin(πy)sin(πz) → (-Δ+k^2)u = (3π^2+k^2)u
u_exact(x, y, z, k) = sin(π*x) * sin(π*y) * sin(π*z)
f_func(x, y, z, k)  = (3π^2 + k^2) * sin(π*x) * sin(π*y) * sin(π*z)


f_func (generic function with 1 method)

In [3]:
# ============================================================
# Jacobi diagonal for the exact-boundary full-grid system
#
# A = H K H' + Sigma + jitter I
# ============================================================

function exact_boundary_jacobi_diag(
    Nx,
    h,
    k_wave,
    is_boundary;
    ν,
    ℓ,
    σf,
    C,
    τ,
    jitter=1e-10
)

    nside = Nx + 2
    N = nside^3

    # Boundary row is identity: H_i = e_i'
    diag_boundary =
        σf^2 + jitter

    # Interior 7-point Helmholtz stencil
    c0 = 6.0/h^2 + k_wave^2
    c1 = -1.0/h^2

    coeffs = [
        c0,
        c1, c1,
        c1, c1,
        c1, c1
    ]

    pts = [
        [0.0,  0.0,  0.0],
        [-h,   0.0,  0.0],
        [ h,   0.0,  0.0],
        [0.0,  -h,   0.0],
        [0.0,   h,   0.0],
        [0.0,  0.0,  -h],
        [0.0,  0.0,   h]
    ]

    Kloc = [
        matern_kernel_3d(
            pts[a],
            pts[b];
            ν=ν,
            ℓ=ℓ,
            σf=σf
        )
        for a in 1:7,
            b in 1:7
    ]

    noise_var =
        C * h^(2τ)

    diag_interior =
        dot(
            coeffs,
            Kloc * coeffs
        ) +
        noise_var +
        jitter

    diagA = [
        is_boundary[q] ?
        diag_boundary :
        diag_interior
        for q in 1:N
    ]

    return diagA
end


exact_boundary_jacobi_diag (generic function with 1 method)

In [4]:
# ============================================================
# Exact-boundary linear algebra benchmark
# with dense/FFT CG and Jacobi-preconditioned variants
# ============================================================

ν      = 1.5
ℓ      = 0.333333
σf     = 1.02239
C      = 327.332
τ      = 2
k_wave = 1.0

dense_max_N = 50000

grid_sizes = [
    3, 4, 5, 6,
    8, 10, 12, 15, 20
]

results = NamedTuple[]

println("===========================================================")
println("Exact-boundary linear algebra benchmark")
println("with Jacobi preconditioning")
println("===========================================================")
@printf("nu      = %.6g\n", ν)
@printf("ell     = %.6g\n", ℓ)
@printf("sigma_f = %.6g\n", σf)
@printf("C       = %.6g\n", C)
@printf("tau     = %d\n", τ)
println()

for Nx in grid_sizes

    h = 1.0 / (Nx + 1)
    nside = Nx + 2
    N = nside^3

    g = collect(
        range(
            0.0,
            1.0,
            length=nside
        )
    )

    X_all = [
        [x,y,z]
        for z in g
        for y in g
        for x in g
    ]

    H, is_boundary =
        build_H_3d_helmholtz_full(
            Nx,
            h,
            k_wave
        )

    b_vec =
        make_y_3d_full(
            X_all,
            k_wave,
            is_boundary
        )

    u_true = [
        u_exact(
            x,
            y,
            z,
            k_wave
        )
        for z in g
        for y in g
        for x in g
    ]

    noise_var =
        C * h^(2τ)

    Sigma_diag = [
        is_boundary[q] ?
        0.0 :
        noise_var
        for q in 1:N
    ]

    jitter = 1e-10

    # --------------------------------------------------------
    # FFT covariance setup
    # --------------------------------------------------------
    k_col_arr = [
        matern_kernel_3d(
            [g[i],g[j],g[l]],
            [g[1],g[1],g[1]];
            ν=ν,
            ℓ=ℓ,
            σf=σf
        )
        for i in 1:nside,
            j in 1:nside,
            l in 1:nside
    ]

    k_fft3 =
        btttb_fft_setup(
            k_col_arr,
            nside,
            nside,
            nside
        )

    # --------------------------------------------------------
    # Matrix-free A
    # --------------------------------------------------------
    A_map = LinearMap(
        v -> begin
            w1 = H' * v
            w2 = btttb_matvec(
                k_fft3,
                w1,
                nside,
                nside,
                nside
            )
            w3 = H * w2
            w3 .+
            Sigma_diag .* v .+
            jitter .* v
        end,
        N;
        issymmetric=true,
        isposdef=true
    )

    # --------------------------------------------------------
    # Jacobi preconditioner
    # --------------------------------------------------------
    diagA =
        exact_boundary_jacobi_diag(
            Nx,
            h,
            k_wave,
            is_boundary;
            ν=ν,
            ℓ=ℓ,
            σf=σf,
            C=C,
            τ=τ,
            jitter=jitter
        )

    M_jacobi =
        Diagonal(diagA)

    # ========================================================
    # 1) Dense Cholesky
    # ========================================================
    if N <= dense_max_N

        t_direct = @elapsed begin

            K = [
                matern_kernel_3d(
                    X_all[i],
                    X_all[j];
                    ν=ν,
                    ℓ=ℓ,
                    σf=σf
                )
                for i in 1:N,
                    j in 1:N
            ]

            A_dense =
                H*K*H' +
                Diagonal(Sigma_diag) +
                jitter*I

            α_dir =
                cholesky(
                    Hermitian(A_dense)
                ) \ b_vec

            μ_dir =
                K * (
                    H' * α_dir
                )
        end

        mse_direct =
            mean(
                (
                    μ_dir .-
                    u_true
                ).^2
            )

    else
        t_direct = NaN
        mse_direct = NaN
        μ_dir = nothing
        A_dense = nothing
        K = nothing
    end

    # ========================================================
    # 2) Dense CG
    # ========================================================
    if N <= dense_max_N

        # reuse A_dense and K from direct branch
        t_cg_dense = @elapsed begin

            α_dense,
            hist_dense =
                cg(
                    Hermitian(A_dense),
                    b_vec;
                    log=true,
                    maxiter=10N,
                    reltol=1e-10
                )

            μ_dense =
                K * (
                    H' * α_dense
                )
        end

        n_iter_dense =
            hist_dense.iters

        mse_dense =
            mean(
                (
                    μ_dense .-
                    u_true
                ).^2
            )

        # ====================================================
        # 3) Dense PCG (Jacobi)
        # ====================================================
        t_pcg_dense = @elapsed begin

            α_pcg_dense,
            hist_pcg_dense =
                cg(
                    Hermitian(A_dense),
                    b_vec;
                    Pl=M_jacobi,
                    log=true,
                    maxiter=10N,
                    reltol=1e-10
                )

            μ_pcg_dense =
                K * (
                    H' * α_pcg_dense
                )
        end

        n_iter_pcg_dense =
            hist_pcg_dense.iters

        mse_pcg_dense =
            mean(
                (
                    μ_pcg_dense .-
                    u_true
                ).^2
            )

    else
        t_cg_dense = NaN
        t_pcg_dense = NaN
        n_iter_dense = -1
        n_iter_pcg_dense = -1
        mse_dense = NaN
        mse_pcg_dense = NaN
    end

    # ========================================================
    # 4) FFT-CG
    # ========================================================
    t_cg_fft = @elapsed begin

        α_fft,
        hist_fft =
            cg(
                A_map,
                b_vec;
                log=true,
                maxiter=10N,
                reltol=1e-10
            )

        μ_fft =
            btttb_matvec(
                k_fft3,
                H' * α_fft,
                nside,
                nside,
                nside
            )
    end

    n_iter_fft =
        hist_fft.iters

    mse_fft =
        mean(
            (
                μ_fft .-
                u_true
            ).^2
        )

    # ========================================================
    # 5) FFT-PCG (Jacobi)
    # ========================================================
    t_pcg_fft = @elapsed begin

        α_pcg_fft,
        hist_pcg_fft =
            cg(
                A_map,
                b_vec;
                Pl=M_jacobi,
                log=true,
                maxiter=10N,
                reltol=1e-10
            )

        μ_pcg_fft =
            btttb_matvec(
                k_fft3,
                H' * α_pcg_fft,
                nside,
                nside,
                nside
            )
    end

    n_iter_pcg_fft =
        hist_pcg_fft.iters

    mse_pcg_fft =
        mean(
            (
                μ_pcg_fft .-
                u_true
            ).^2
        )

    rel_diff =
        μ_dir === nothing ?
        NaN :
        norm(
            μ_dir -
            μ_pcg_fft
        ) /
        max(
            norm(μ_dir),
            eps()
        )

    push!(
        results,
        (
            Nx=Nx,
            N=N,
            h=h,
            noise_var=noise_var,
            t_direct=t_direct,
            t_cg_dense=t_cg_dense,
            t_pcg_dense=t_pcg_dense,
            t_cg_fft=t_cg_fft,
            t_pcg_fft=t_pcg_fft,
            n_iter_dense=n_iter_dense,
            n_iter_pcg_dense=n_iter_pcg_dense,
            n_iter_fft=n_iter_fft,
            n_iter_pcg_fft=n_iter_pcg_fft,
            mse_direct=mse_direct,
            mse_dense=mse_dense,
            mse_pcg_dense=mse_pcg_dense,
            mse_fft=mse_fft,
            mse_pcg_fft=mse_pcg_fft,
            rel_diff=rel_diff
        )
    )

    chol_str =
        isnan(t_direct) ?
        "skip" :
        @sprintf("%.3gs",t_direct)

    dense_str =
        isnan(t_cg_dense) ?
        "skip" :
        @sprintf(
            "%.3gs(%d)",
            t_cg_dense,
            n_iter_dense
        )

    pdense_str =
        isnan(t_pcg_dense) ?
        "skip" :
        @sprintf(
            "%.3gs(%d)",
            t_pcg_dense,
            n_iter_pcg_dense
        )

    @printf(
        "Nx=%2d N=%6d | Chol=%-10s | CGdense=%-14s | PCGdense=%-14s | FFT-CG=%.3gs(%d) | FFT-PCG=%.3gs(%d)\n",
        Nx,
        N,
        chol_str,
        dense_str,
        pdense_str,
        t_cg_fft,
        n_iter_fft,
        t_pcg_fft,
        n_iter_pcg_fft
    )
end


Exact-boundary linear algebra benchmark
with Jacobi preconditioning
nu      = 1.5
ell     = 0.333333
sigma_f = 1.02239
C       = 327.332
tau     = 2

Nx= 3 N=   125 | Chol=0.356s     | CGdense=0.226s(23)     | PCGdense=0.092s(12)     | FFT-CG=0.319s(22) | FFT-PCG=0.0844s(12)
Nx= 4 N=   216 | Chol=0.0107s    | CGdense=0.000268s(36)  | PCGdense=7.3e-05s(13)   | FFT-CG=0.00149s(33) | FFT-PCG=0.000674s(13)
Nx= 5 N=   343 | Chol=0.0239s    | CGdense=0.000596s(113) | PCGdense=0.000234s(36)  | FFT-CG=0.0116s(113) | FFT-PCG=0.00333s(36)
Nx= 6 N=   512 | Chol=0.0498s    | CGdense=0.00235s(150)  | PCGdense=0.000448s(37)  | FFT-CG=0.0137s(150) | FFT-PCG=0.00297s(38)
Nx= 8 N=  1000 | Chol=0.21s      | CGdense=0.0156s(533)   | PCGdense=0.00386s(96)   | FFT-CG=0.074s(525) | FFT-PCG=0.0158s(94)
Nx=10 N=  1728 | Chol=0.575s     | CGdense=0.0899s(1321)  | PCGdense=0.0136s(200)   | FFT-CG=0.457s(1377) | FFT-PCG=0.0615s(203)
Nx=12 N=  2744 | Chol=1.32s      | CGdense=0.417s(2579)   | PCGdense=0.0641s(363

In [ ]:
# ============================================================
# Summary table
# ============================================================

println()
println("=== Preconditioning comparison ===")

@printf(
    "%-4s %-7s %-10s %-10s %-10s %-10s %-9s %-9s %-9s %-9s\n",
    "Nx","N",
    "CGdense","PCGdense",
    "FFT-CG","FFT-PCG",
    "itCd","itPCd",
    "itCf","itPCf"
)

println("-"^98)

for r in results

    d1 =
        isnan(r.t_cg_dense) ?
        "skip" :
        @sprintf("%.4g",r.t_cg_dense)

    d2 =
        isnan(r.t_pcg_dense) ?
        "skip" :
        @sprintf("%.4g",r.t_pcg_dense)

    @printf(
        "%-4d %-7d %-10s %-10s %-10.4g %-10.4g %-9d %-9d %-9d %-9d\n",
        r.Nx,
        r.N,
        d1,
        d2,
        r.t_cg_fft,
        r.t_pcg_fft,
        r.n_iter_dense,
        r.n_iter_pcg_dense,
        r.n_iter_fft,
        r.n_iter_pcg_fft
    )
end


In [ ]:
# ============================================================
# Timing comparison
# ============================================================

Ns =
    [r.N for r in results]

t_cg_dense =
    [r.t_cg_dense for r in results]

t_pcg_dense =
    [r.t_pcg_dense for r in results]

t_cg_fft =
    [r.t_cg_fft for r in results]

t_pcg_fft =
    [r.t_pcg_fft for r in results]

mask_dense =
    .!isnan.(t_cg_dense)

p_time = plot(
    Ns[mask_dense],
    t_cg_dense[mask_dense];
    marker=:circle,
    lw=2,
    label="CG dense",
    xlabel="N=(Nx+2)^3",
    ylabel="Time (s)",
    title="Exact-boundary CG/PCG timing",
    framestyle=:box
)

plot!(
    p_time,
    Ns[mask_dense],
    t_pcg_dense[mask_dense];
    marker=:diamond,
    lw=2,
    label="Jacobi PCG dense"
)

plot!(
    p_time,
    Ns,
    t_cg_fft;
    marker=:square,
    lw=2,
    label="CG FFT"
)

plot!(
    p_time,
    Ns,
    t_pcg_fft;
    marker=:star5,
    lw=2,
    label="Jacobi PCG FFT"
)

display(p_time)

savefig(
    p_time,
    "helmholtz3d_exact_boundary_preconditioning_timing.png"
)


In [ ]:
# ============================================================
# Iteration-count comparison
# ============================================================

it_cg_dense =
    [r.n_iter_dense for r in results]

it_pcg_dense =
    [r.n_iter_pcg_dense for r in results]

it_cg_fft =
    [r.n_iter_fft for r in results]

it_pcg_fft =
    [r.n_iter_pcg_fft for r in results]

p_iter = plot(
    Ns[mask_dense],
    it_cg_dense[mask_dense];
    marker=:circle,
    lw=2,
    label="CG dense",
    xlabel="N=(Nx+2)^3",
    ylabel="Iterations",
    title="Exact-boundary CG iteration count",
    framestyle=:box
)

plot!(
    p_iter,
    Ns[mask_dense],
    it_pcg_dense[mask_dense];
    marker=:diamond,
    lw=2,
    label="Jacobi PCG dense"
)

plot!(
    p_iter,
    Ns,
    it_cg_fft;
    marker=:square,
    lw=2,
    label="CG FFT"
)

plot!(
    p_iter,
    Ns,
    it_pcg_fft;
    marker=:star5,
    lw=2,
    label="Jacobi PCG FFT"
)

display(p_iter)

savefig(
    p_iter,
    "helmholtz3d_exact_boundary_preconditioning_iterations.png"
)
